Trying to automatically update the GFU dataset

In [1]:
%cd '/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series'

/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series


/opt/anaconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import pandas as pd
import openpyxl

In [3]:
# 1. Load the entire Excel workbook
# Setting sheet_name=None tells Python to grab EVERY sheet at once.
file_path = 'No_DS_10YR_RETURNS_DEL_COL_update2026.xlsx'
all_sheets = pd.read_excel(file_path, sheet_name=None)

# Creating an empty datafram that will be filld by the following structure: Date (Daily), Australia, Austria...
daily_df = pd.DataFrame()

Now I want to create a cycle in wich the Python extracts just the raw data and compute the daily return * 100, for each country.

For doing so, I need to select just the specific column A and B, but from row 8 towards

In [4]:
for country_name, df in all_sheets.items():
    print(f"Extracting daily data from: {country_name}")

    try:
        #selecting the data that I need. we skip the first 8 columns of Datastream junk and grab columns 2 (Dates) and 3 (Values)
        df_clean = df.iloc[8:, [2,3]].copy()

        # We want to return column with the actual country name
        df_clean.columns = ["Date", country_name]

        # Clean the data to ensure dates and numbers are recognised properly
        df_clean["Date"]       = pd.to_datetime(df_clean["Date"], errors = "coerce")
        df_clean[country_name] = pd.to_numeric(df_clean[country_name], errors = "coerce")
        df_clean               = df_clean.dropna(subset=["Date", country_name])

        #We set the Date as the "Index"
        df_clean.set_index('Date', inplace=True)

        #Now we merge this country's daily returns inot the aster table
        if daily_df.empty:
            daily_df = df_clean
        else:
            # An 'outer' join is crucial here: it perfectly aligns the dates.
            # If Australia has a public holiday but Austria doesn't, Python aligns the days correctly!
            daily_df = daily_df.join(df_clean, how='outer')
    
    except Exception as e:
        print(f"Could not process {country_name}. Error: {e}")

print("\nStep 1 Complete! Here is your Daily Master DataFrame:")

Extracting daily data from: Australia
Extracting daily data from: Austria
Extracting daily data from: Belgium
Extracting daily data from: Brazil
Extracting daily data from: Canada
Extracting daily data from: Chile
Extracting daily data from: China
Extracting daily data from: Colombia
Extracting daily data from: CZ
Extracting daily data from: Denmark
Extracting daily data from: Egypt
Extracting daily data from: Finland
Extracting daily data from: France
Extracting daily data from: Germany
Extracting daily data from: Greece
Extracting daily data from: Hong Kong
Extracting daily data from: Hungary
Extracting daily data from: India
Extracting daily data from: Indonesia
Extracting daily data from: Ireland
Extracting daily data from: Israel
Extracting daily data from: Italy
Extracting daily data from: Japan
Extracting daily data from: Korea
Extracting daily data from: Malaysia
Extracting daily data from: Mexico
Extracting daily data from: Netherlands
Extracting daily data from: Nigeria
Extra

/var/folders/_7/z6fx6jdx1h90fmhzn2dtnxfr0000gn/T/ipykernel_84591/2023099276.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Date"]       = pd.to_datetime(df_clean["Date"], errors = "coerce")
/var/folders/_7/z6fx6jdx1h90fmhzn2dtnxfr0000gn/T/ipykernel_84591/2023099276.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Date"]       = pd.to_datetime(df_clean["Date"], errors = "coerce")


In [5]:
# Sort the dates from oldest to newest, just as you requested
daily_df = daily_df.sort_index()

# Display to confirm the structure matches your drawing
daily_df

,Australia,Austria,Belgium,Brazil,Canada,Chile,China,Colombia,CZ,Denmark,...,Spain,Sweden,Taiwan,Switzerland,Thailand,Turkey,UK,US,Venezuela,SA
Date,,,,,,,,,,,,,,,,,,,,,
1957-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-02-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-02-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,-0.283803,0.361337,0.573394,0.365764,1.266552,-0.651163,-0.280899,-0.606777,NaN,-0.783379,...,0.657707,-0.798548,0.0,-6.633907,0.000000,-0.333556,0.602285,0.724130,NaN,-0.118977
2026-04-16,0.724492,NaN,NaN,0.400729,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.344333,0.072543,0.0,4.009434,-1.980198,0.266134,0.783021,0.649803,NaN,0.767414
2026-04-17,NaN,NaN,NaN,-0.978517,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Save the master table as a true Excel Workbook
file_name = 'PHASE_B_10YR_monthly_df.xlsx'
daily_df.to_excel(file_name)

print(f"Success! Your data has been saved as '{file_name}' in your project folder.")

Success! Your data has been saved as 'PHASE_B_10YR_monthly_df.xlsx' in your project folder.


Good news, the code works greatly.

## The following piece of code is to complete and to verify
I would now like to update the input (raw) excel file with the computed daily return that we can extract from the daily_df
It is the exact opposit thing that we have done for computing the daily_df

In [7]:
import pandas as pd
import openpyxl

print("Loading the raw Excel workbook. This might take a minute for a large file...")

# 1. Load the original workbook (this preserves all your sheets and Datastream headers)
file_path = 'No_DS_10YR_RETURNS_DEL_COL_update2026.xlsx'
wb = openpyxl.load_workbook(file_path)

# 2. Loop through every sheet in the workbook
for sheet_name in wb.sheetnames:
    print(f"Updating sheet: {sheet_name}")
    sheet = wb[sheet_name]
    
    # Only process countries that are actually columns in our daily_master table
    if sheet_name in daily_df.columns:
        
        # 3. Start reading from Excel Row 9 (where the actual dates and returns begin)
        for row in range(9, sheet.max_row + 1):
            
            # Grab the date from Column C (which is column 3 in openpyxl)
            cell_date = sheet.cell(row=row, column=3).value
            
            if pd.notna(cell_date):
                try:
                    # Convert the Excel date into the same 'YYYY-MM-DD' string format we used for our index
                    date_str = pd.to_datetime(cell_date).strftime('%Y-%m-%d')
                    
                    # Check if this date exists in our master dataframe
                    if date_str in daily_df.index:
                        
                        # Extract the computed return for this specific country and date
                        computed_return = daily_df.at[date_str, sheet_name]
                        
                        # If it is a valid number, write it back into Column D (column 4)
                        if pd.notna(computed_return):
                            sheet.cell(row=row, column=4).value = computed_return
                            
                except Exception:
                    # If a row has weird text instead of a date, simply skip it
                    pass

# 4. Save it as a brand new file so we do not overwrite your original raw data
new_file_name = 'UPDATED_No_DS_10YR_RETURNS.xlsx'
wb.save(new_file_name)
print(f"\nSuccess! Your fully updated Excel file has been saved as '{new_file_name}'.")

Loading the raw Excel workbook. This might take a minute for a large file...


/opt/anaconda3/lib/python3.12/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


KeyboardInterrupt: 

## PHASE B

Now I want to compute the MONTHLY standard deviation and skewness per each country from the daily_df creating the montly_df

In [8]:
print("Calculating monthly statistics from daily_df...")

# 1. Resample to Monthly Frequency and Calculate SD & SK
# 'MS' stands for Month Start. We tell pandas to group the daily dates into months,
# and calculate both 'std' (Standard Deviation) and 'skew' simultaneously.
monthly_df = daily_df.resample('MS').agg(['std', 'skew'])

# 2. Flatten the Column Names
# Right now, the columns are stacked in two levels (e.g., 'Australia' on top, 'std' underneath).
# We to flatten them so that we obtain (e.g., 'SD Australia', 'SK Australia', ..., 'SD Italy', 'SK Italy').
new_columns = []
for country, stat in monthly_df.columns:
    if stat == 'std':
        new_columns.append(f'SD {country}')
    elif stat == 'skew':
        new_columns.append(f'SK {country}')

# Apply the new clean column names
monthly_df.columns = new_columns

print("\nStep 2 Complete! Here is your final Monthly DataFrame:")
monthly_df

Calculating monthly statistics from daily_df...

Step 2 Complete! Here is your final Monthly DataFrame:


,SD Australia,SK Australia,SD Austria,SK Austria,SD Belgium,SK Belgium,SD Brazil,SK Brazil,SD Canada,SK Canada,...,SD Turkey,SK Turkey,SD UK,SK UK,SD US,SK US,SD Venezuela,SK Venezuela,SD SA,SK SA
Date,,,,,,,,,,,,,,,,,,,,,
1957-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-01,0.655607,-1.170835,0.948888,0.121280,0.968981,0.348668,0.774692,1.340445,1.406083,1.527387,...,0.503757,-0.910852,0.584650,0.433290,0.757868,0.323547,NaN,NaN,0.610772,0.661036
2026-01-01,0.674540,-1.906020,1.114303,1.421122,0.735074,0.449770,0.594333,-1.207560,0.763513,0.043800,...,0.625592,0.422729,0.800726,-0.238263,0.683138,0.472640,NaN,NaN,0.718901,-0.290357
2026-02-01,0.850941,0.298164,0.540724,-0.446027,0.470856,-0.168802,0.444418,0.625050,0.643706,-0.121450,...,0.547779,0.066716,0.539851,0.185197,0.855508,-0.541439,NaN,NaN,0.754488,-0.179197


In [9]:
# Sort the dates from oldest to newest, just as you requested
monthly_df = monthly_df.sort_index()

# Display to confirm the structure matches your drawing
monthly_df

,SD Australia,SK Australia,SD Austria,SK Austria,SD Belgium,SK Belgium,SD Brazil,SK Brazil,SD Canada,SK Canada,...,SD Turkey,SK Turkey,SD UK,SK UK,SD US,SK US,SD Venezuela,SK Venezuela,SD SA,SK SA
Date,,,,,,,,,,,,,,,,,,,,,
1957-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1957-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-01,0.655607,-1.170835,0.948888,0.121280,0.968981,0.348668,0.774692,1.340445,1.406083,1.527387,...,0.503757,-0.910852,0.584650,0.433290,0.757868,0.323547,NaN,NaN,0.610772,0.661036
2026-01-01,0.674540,-1.906020,1.114303,1.421122,0.735074,0.449770,0.594333,-1.207560,0.763513,0.043800,...,0.625592,0.422729,0.800726,-0.238263,0.683138,0.472640,NaN,NaN,0.718901,-0.290357
2026-02-01,0.850941,0.298164,0.540724,-0.446027,0.470856,-0.168802,0.444418,0.625050,0.643706,-0.121450,...,0.547779,0.066716,0.539851,0.185197,0.855508,-0.541439,NaN,NaN,0.754488,-0.179197


In [11]:
# Save the master table as a true Excel Workbook
file_name = 'PHASE_B_10YR_monthly_df.xlsx'
monthly_df.to_excel(file_name)

print(f"Success! Your data has been saved as '{file_name}' in your project folder.")

Success! Your data has been saved as 'PHASE_B_10YR_monthly_df.xlsx' in your project folder.
